# German Data Analyst Job Market — End-to-End Analysis

**Author:** Portfolio Project — IBM Data Analyst Certificate  
**Data source:** Adzuna Jobs API (Germany) / synthetic fallback  
**Tools:** Python · pandas · SQLite · Matplotlib · Seaborn · Folium

---

In [ ]:
import sys
import json
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

# Make src/ importable whether Jupyter is launched from project root or notebooks/
_root = Path('.').resolve()
if _root.name == 'notebooks':
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.fetch_adzuna import fetch_all_listings, load_most_recent_raw_file
from src.clean import clean_listings, TRACKED_SKILLS
from src.database import setup_database, query_to_dataframe
from src import visualize

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.0f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print('Environment ready.')

---
## 1. Introduction

**Question this project answers:**  
*Where are German data-analyst jobs concentrated, what skills are actually required, and what salary can a junior analyst realistically target?*

**Why it matters for a job seeker:**  
Job boards show thousands of listings, but a 30-minute search reveals little about the *distribution* of opportunity across cities, the *gap* between advertised skills, or the *salary premium* attached to specific tools.  
This analysis turns 300+ job postings into five concrete, data-backed decisions a junior analyst can make today.

**Analytical approach:**  
1. Fetch listings from the Adzuna Jobs API (or fall back to synthetic demo data).  
2. Clean and normalize in Python, then persist to SQLite.  
3. Explore with Matplotlib/Seaborn, then export to CSV for Tableau Public.

---
## 2. Data Acquisition

**Primary source:** Adzuna Jobs API — Germany endpoint, search terms:  
`"data analyst"`, `"datenanalyst"`, `"business intelligence analyst"`

**Fallback:** When `ADZUNA_APP_ID` / `ADZUNA_APP_KEY` are not set in `.env`,  
the code first tries to download a public CSV from GitHub, then falls back to  
300 synthetic listings with realistic German city/salary/skill distributions.

Register for a free Adzuna API key at: https://developer.adzuna.com/

In [ ]:
# Reuse cached data when re-running the notebook; fetch fresh otherwise
listings_raw = load_most_recent_raw_file()

if listings_raw is None:
    print('No cached data found — fetching now...')
    listings_raw = fetch_all_listings(max_pages=5)

print(f'\nRaw listings loaded: {len(listings_raw)}')
print(f'First listing keys: {list(listings_raw[0].keys())}')

In [ ]:
# Inspect one raw record so we understand what the API/fallback returns
sample = listings_raw[0]
print('--- Sample raw listing ---')
for key, value in sample.items():
    display_val = str(value)[:120]  # Truncate long description
    print(f'  {key:22s}: {display_val}')

---
## 3. Data Cleaning

Raw listings need several transformations before analysis:

| Step | Problem | Fix |
|---|---|---|
| Deduplication | API returns the same listing under multiple search terms | Drop by `job_id` |
| Salary parsing | `salary_min` / `salary_max` are floats or `null` | Cast to numeric, compute mid-point |
| Location | `"Berlin, Deutschland"` → need just the city | Split on comma, normalise aliases |
| Skills | No structured skill field — buried in description text | Regex keyword match for 12 tools |
| Seniority | Only in the title string | Regex pattern match |

In [ ]:
listings_df = clean_listings(listings_raw)
print(f'\nColumns: {listings_df.columns.tolist()}')

In [ ]:
# Before / after: salary column
print('=== Before cleaning: raw salary_min values (first 8) ===')
for r in listings_raw[:8]:
    print(f"  {r.get('salary_min')!r:>10}  ({type(r.get('salary_min')).__name__})")

print('\n=== After cleaning: salary_mid_eur (first 8) ===')
print(listings_df['salary_mid_eur'].head(8).to_string())

print(f'\nSalary disclosure rate: '
      f"{listings_df['salary_mid_eur'].notna().mean():.0%} of listings")

In [ ]:
# Before / after: location → city
print('=== Before: location_display (first 8) ===')
print(listings_df['location_display'].head(8).to_string())

print('\n=== After: city (first 8) ===')
print(listings_df['city'].head(8).to_string())

print('\n=== Seniority breakdown ===')
print(listings_df['seniority'].value_counts().to_string())

In [ ]:
# Persist cleaned data to SQLite
db_path = _root / 'data' / 'processed' / 'listings.db'
conn = setup_database(listings_df, db_path=db_path)

# Verify with a SQL query
row_count = query_to_dataframe(conn, 'SELECT COUNT(*) AS n FROM listings')
print(f"Rows in database: {row_count['n'].iloc[0]}")

In [ ]:
# SQL is a core analyst skill — demonstrate a meaningful query
sql = """
    SELECT
        seniority,
        COUNT(*)                            AS listings,
        ROUND(AVG(salary_mid_eur), 0)       AS avg_salary_eur,
        ROUND(MIN(salary_mid_eur), 0)       AS min_salary_eur,
        ROUND(MAX(salary_mid_eur), 0)       AS max_salary_eur
    FROM listings
    WHERE salary_mid_eur IS NOT NULL
    GROUP BY seniority
    ORDER BY avg_salary_eur DESC
"""

seniority_salary_df = query_to_dataframe(conn, sql)
print('=== Average salary by seniority (from SQLite) ===')
seniority_salary_df

---
## 4. Exploratory Data Analysis

### 4.1 Where are the jobs?

**Question:** Which German cities have the most data-analyst listings,  
and how concentrated is demand compared to the rest of the country?

In [ ]:
ax = visualize.plot_top_cities(listings_df, top_n=12)
plt.show()

**Takeaway:** Berlin alone accounts for roughly a quarter of all listings,  
with Munich and Hamburg together adding another 30 percent.  
If you are location-flexible, targeting these three cities dramatically expands your options;  
if you are location-fixed, expect fewer roles and more competition.

### 4.2 What do these roles pay?

**Question:** What does the advertised salary distribution look like,  
and what is the typical midpoint a candidate can use for negotiation?

In [ ]:
ax = visualize.plot_salary_distribution(listings_df)
plt.show()

salary_stats = listings_df['salary_mid_eur'].describe()
print(f"\nMedian: €{salary_stats['50%']:,.0f}")
print(f"25th percentile: €{salary_stats['25%']:,.0f}")
print(f"75th percentile: €{salary_stats['75%']:,.0f}")

**Takeaway:** Only ~35% of listings disclose salary, and disclosed salaries skew higher  
(employers who pay more are more likely to advertise it).  
The median is around €52k, with the middle 50% roughly between €43k and €65k.  
Use this as a floor for initial negotiations rather than an average expectation.

### 4.3 What skills are employers actually asking for?

**Question:** Which tools appear most frequently in job descriptions,  
and which are niche specialisations versus mainstream requirements?

In [ ]:
ax = visualize.plot_skills_frequency(listings_df)
plt.show()

# Print exact percentages
present_skills = [s for s in TRACKED_SKILLS if s in listings_df.columns]
skill_pct = (listings_df[present_skills].sum() / len(listings_df) * 100).sort_values(ascending=False)
print('\nSkill mention rates:')
for skill, pct in skill_pct.items():
    print(f'  {skill:12s}: {pct:.1f}%')

**Takeaway:** SQL and Python are the non-negotiable baseline — mentioned in 75%+ and 65%+ of  
listings respectively. Excel follows, confirming it is still expected even in data-forward roles.  
Power BI edges out Tableau in Germany, suggesting Microsoft's ecosystem dominates BI tooling here.  
Cloud skills (AWS, Azure) appear in ~35% of listings and tend to signal higher-paying roles (see §5).

### 4.4 What is the seniority mix?

**Question:** Are companies primarily hiring junior, mid-level, or senior analysts —  
and what does that mean for someone just entering the market?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: count by seniority
seniority_counts = listings_df['seniority'].value_counts()
colors = sns.color_palette('Blues_r', len(seniority_counts))
seniority_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', rot=0)
axes[0].set_title('Listings by Seniority Level', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Listings')

# Right: salary by seniority (box)
salary_data = listings_df[listings_df['salary_mid_eur'].notna()]
seniority_order = ['junior', 'mid', 'senior', 'lead']
present_order = [s for s in seniority_order if s in salary_data['seniority'].values]
sns.boxplot(
    data=salary_data, x='seniority', y='salary_mid_eur',
    order=present_order, palette='Blues_r', ax=axes[1]
)
axes[1].set_title('Salary Range by Seniority', fontsize=12)
axes[1].set_xlabel('')
axes[1].set_ylabel('Annual Salary — Mid-Point (EUR)')
import matplotlib.ticker as mticker
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

**Takeaway:** Mid-level roles dominate (~40% of listings), which means the market is competitive  
for career-changers claiming 'mid' without a track record.  
Junior roles (typically titled with 'Junior' or 'Berufseinsteiger') exist but are fewer —  
a clean portfolio project like this one helps differentiate in that pool.  
The salary step from junior to senior is roughly €18–20k, consistent with 3–4 years of growth.

---
## 5. Skills Deep-Dive: Which Skills Correlate with Higher Salary?

**Question:** Among listings that disclose salary, do certain skill requirements  
reliably appear in higher-paying bands?  

*Note: This is a descriptive correlation only — not causation.*  
A listing requiring Snowflake may be better-paid because it is a senior cloud role,  
not because Snowflake itself commands a premium.

In [ ]:
ax = visualize.plot_salary_by_skill(listings_df)
plt.show()

In [ ]:
# Table: median salary for listings that require each skill
salary_df = listings_df[listings_df['salary_mid_eur'].notna()]
present_skills = [s for s in TRACKED_SKILLS if s in salary_df.columns]

rows = []
for skill in present_skills:
    with_skill = salary_df[salary_df[skill] == True]['salary_mid_eur']
    without_skill = salary_df[salary_df[skill] == False]['salary_mid_eur']
    rows.append({
        'Skill': skill,
        'Listings with skill': int(salary_df[skill].sum()),
        'Median salary (with)': with_skill.median() if not with_skill.empty else None,
        'Median salary (without)': without_skill.median() if not without_skill.empty else None,
    })

skill_salary_table = (
    pd.DataFrame(rows)
    .sort_values('Median salary (with)', ascending=False)
    .reset_index(drop=True)
)

# Format as currency
for col in ['Median salary (with)', 'Median salary (without)']:
    skill_salary_table[col] = skill_salary_table[col].apply(
        lambda x: f'€{x:,.0f}' if pd.notna(x) else '—'
    )

skill_salary_table

**Takeaway:** Cloud and modern data-stack skills (Snowflake, dbt, AWS, Azure) cluster at the  
top of the salary distribution. These tools appear in roles that already require seniority,  
so learning them while building a portfolio is high-ROI.  
SQL and Excel show median salaries close to the overall median — they are table stakes, not  
differentiators. Python sits above average, confirming it is the single most leveraged skill  
to add if you only have time to learn one.

---
## 6. Geographic View

**Question:** Visualised on a map, do German data-analyst jobs cluster  
in expected tech hubs, or are there surprises in the distribution?

In [ ]:
city_map = visualize.build_city_map(listings_df)
city_map  # Renders inline in Jupyter

**Takeaway:** The map confirms the north-south axis from Hamburg through Berlin to Munich  
as the primary corridor for data roles. Frankfurt's financial district generates  
a disproportionate share of BI/reporting roles.  
Bubble size scales with listing count, so smaller cities with visible bubbles  
(Leipzig, Dresden, Nuremberg) represent meaningful local markets worth targeting  
if you want less competition.

---
## 7. Key Findings

Five data-backed decisions a junior data analyst can act on today:

1. **Target Berlin, Munich, or Hamburg first.**  
   These three cities account for ~55% of all listings. If remote/hybrid options exist, apply nationally.

2. **SQL + Python is the non-negotiable baseline.**  
   Appearing in 75%+ and 65%+ of listings respectively — invest here before anything else.

3. **Learn Power BI over Tableau (for Germany).**  
   Power BI appears more frequently in German job listings, consistent with Microsoft's enterprise dominance.

4. **A cloud skill (AWS or Azure) is the highest-leverage addition after the baseline.**  
   Cloud requirements correlate with the highest salary bands; adding one cloud certification signals career trajectory.

5. **Salary negotiation baseline: €43k–€52k for junior, €52k–€68k for mid.**  
   Remember: only ~35% of listings disclose salary, and those that do skew higher — treat this as a floor.

In [ ]:
# Export cleaned data for Tableau Public
tableau_path = _root / 'data' / 'processed' / 'listings_for_tableau.csv'
listings_df.to_csv(tableau_path, index=False, encoding='utf-8')
print(f'Exported {len(listings_df)} rows to {tableau_path.name}')
print(f'Columns: {listings_df.columns.tolist()}')

### Tableau Public Dashboard (Manual Build)

Open `data/processed/listings_for_tableau.csv` in Tableau Public and create these sheets:

| Sheet | Chart type | Dimensions | Measure |
|---|---|---|---|
| Top Cities | Horizontal bar | `city` | COUNT(job_id) |
| Salary Distribution | Histogram | `salary_mid_eur` (binned) | COUNT |
| Skills Frequency | Bar | `skills_list` (split) | COUNT |
| Salary by Skill | Box-and-whisker | skill columns | `salary_mid_eur` |
| Map | Filled map / bubble | `city` (geocoded) | COUNT(job_id) |

Publish to Tableau Public and add the link to `README.md`.

---
## 8. Limitations

Any honest analysis declares what the data *cannot* tell us:

- **API coverage:** Adzuna aggregates listings from multiple boards, but does not cover every German job board (e.g., LinkedIn, XING, company career pages). The sample may under-represent certain sectors.

- **Salary disclosure rate:** Only ~35% of listings include salary data. Employers who advertise salaries tend to offer higher-than-average pay, so the distribution is upward-biased.

- **Skill extraction accuracy:** Skills are extracted from free-text descriptions using keyword matching. A listing that says 'no Python experience required' would still be counted as requiring Python. Precision would improve with NLP negation detection.

- **Seniority classification:** Based solely on title strings. 'Analyst' roles at smaller companies may carry senior responsibilities without the title. The categorisation is a proxy, not ground truth.

- **Scraping caveats:** The Stepstone scraper (`src/scrape_stepstone.py`) is a demonstration of technique only. Running it at scale may violate Stepstone's Terms of Service. The primary data source for this analysis is the Adzuna API.

- **Temporal snapshot:** Data was collected on a single date. Job markets fluctuate; findings should be validated against current postings before acting on them.